# Modelado Random Forest (2018–2024)

Este notebook forma parte del pipeline de ciencia de datos del proyecto **crash-severity-predictor**.

El objetivo es desarrollar, entrenar y evaluar un modelo **Random Forest** para la predicción de severidad en hechos de tránsito a partir del dataset procesado durante las fases de EDA y ETL. Esta implementación constituye la primera iteración dentro del conjunto de modelos candidatos del proyecto.


In [1]:
# -- Importaciones ----------------------------------------------
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, f1_score, accuracy_score,
                             roc_curve, precision_recall_curve)
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print('✓ Librerías cargadas correctamente')

✓ Librerías cargadas correctamente


## 1. Carga de datos

In [2]:
# -- Carga ----------------------------------------------
train = pd.read_parquet('../data/clean/train.parquet')
test  = pd.read_parquet('../data/clean/test.parquet')

FEATURES = ['tipo_eve','tipo_veh','g_hora_5','dia_sem_ocu',
            'sexo_per','edad_quinquenales','mayor_menor','depto_ocu']
TARGET = 'fall_les'

X_train = train[FEATURES]
y_train = train[TARGET]
X_test  = test[FEATURES]
y_test  = test[TARGET]

print(f'Train : {X_train.shape}')
print(f'Test  : {X_test.shape}')
print(f'\nDistribución target (test):')
print(y_test.value_counts().rename({1:"Fallecido", 2:"Lesionado"}))

Train : (57553, 8)
Test  : (14389, 8)

Distribución target (test):
fall_les
Lesionado    11641
Fallecido     2748
Name: count, dtype: int64


## 2. Entrenamiento Random Forest

In [3]:
# -- Entrenamiento ----------------------------------------------
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
print('✓ Modelo entrenado correctamente')

✓ Modelo entrenado correctamente


## 3. Evaluación del modelo

In [4]:
# -- Predicciones ----------------------------------------------
y_pred  = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]

acc  = accuracy_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred, average='weighted')
auc  = roc_auc_score(y_test, y_proba)
cm   = confusion_matrix(y_test, y_pred)

print('=== Random Forest ===')
print(f'Accuracy : {acc:.4f}')
print(f'F1-Score : {f1:.4f}')
print(f'ROC-AUC  : {auc:.4f}')
print(f'\n{classification_report(y_test, y_pred, target_names=["Fallecido","Lesionado"])}')

=== Random Forest ===
Accuracy : 0.6949
F1-Score : 0.7233
ROC-AUC  : 0.7336

              precision    recall  f1-score   support

   Fallecido       0.34      0.62      0.44      2748
   Lesionado       0.89      0.71      0.79     11641

    accuracy                           0.69     14389
   macro avg       0.61      0.67      0.61     14389
weighted avg       0.78      0.69      0.72     14389



In [5]:
# -- Visualización 1: Métricas generales ----------------------------------------------
fig_metricas = go.Figure(go.Bar(
    x=['Accuracy', 'F1-Score', 'ROC-AUC'],
    y=[acc, f1, auc],
    text=[f'{acc:.4f}', f'{f1:.4f}', f'{auc:.4f}'],
    textposition='outside',
    marker_color=['#5B8DEF', '#F4A261', '#2EC4B6'],
    width=0.4
))
fig_metricas.update_layout(
    title='Métricas de evaluación : Random Forest',
    yaxis=dict(range=[0, 1], title='Valor'),
    xaxis_title='Métrica',
    height=400,
    template='plotly_white'
)
fig_metricas.show()

In [6]:
# -- Visualización 2: Matriz de confusión ----------------------------------------------
cm_labels = ['Fallecido', 'Lesionado']
fig_cm = px.imshow(
    cm,
    labels=dict(x='Predicción', y='Real', color='Cantidad'),
    x=cm_labels,
    y=cm_labels,
    text_auto=True,
    color_continuous_scale='Blues',
    title='Matriz de confusión : Random Forest'
)
fig_cm.update_layout(height=400, template='plotly_white')
fig_cm.show()

In [7]:
# -- Visualización 3: Curva ROC ----------------------------------------------
fpr, tpr, _ = roc_curve(y_test, y_proba, pos_label=1)

fig_roc = go.Figure()
fig_roc.add_trace(go.Scatter(
    x=fpr, y=tpr,
    mode='lines',
    name=f'Random Forest (AUC = {auc:.4f})',
    line=dict(color='#5B8DEF', width=2.5)
))
fig_roc.add_trace(go.Scatter(
    x=[0,1], y=[0,1],
    mode='lines',
    name='Baseline (AUC = 0.5)',
    line=dict(color='gray', width=1.5, dash='dash')
))
fig_roc.update_layout(
    title='Curva ROC : Random Forest',
    xaxis_title='Tasa de Falsos Positivos',
    yaxis_title='Tasa de Verdaderos Positivos',
    height=450,
    template='plotly_white',
    legend=dict(x=0.6, y=0.1)
)
fig_roc.show()

In [8]:
# -- Visualización 4: Importancia de features ----------------------------------------------
importancias = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()

LABELS = {
    'tipo_eve'         : 'Tipo de evento',
    'tipo_veh'         : 'Tipo de vehículo',
    'g_hora_5'         : 'Grupo horario',
    'dia_sem_ocu'      : 'Día de la semana',
    'sexo_per'         : 'Sexo',
    'edad_quinquenales': 'Grupo de edad',
    'mayor_menor'      : 'Mayor / Menor edad',
    'depto_ocu'        : 'Departamento'
}

fig_imp = go.Figure(go.Bar(
    x=importancias.values,
    y=[LABELS[f] for f in importancias.index],
    orientation='h',
    marker_color='#5B8DEF',
    text=[f'{v:.4f}' for v in importancias.values],
    textposition='auto',
    textfont=dict(size=11)
))
fig_imp.update_layout(
    title='Importancia de features : Random Forest',
    xaxis=dict(title='Importancia (Gini)', range=[0, max(importancias.values) * 1.25]),
    yaxis_title='Feature',
    height=450,
    template='plotly_white'
)
fig_imp.show()

### Hallazgo — Importancia de features

| Feature | Importancia |
|---|---|
| Departamento | 0.2342 : más predictora |
| Tipo de evento | 0.1849 |
| Grupo de edad | 0.1655 |
| Tipo de vehículo | 0.1276 |
| Sexo | 0.1104 |
| Día de la semana | 0.0966 |
| Grupo horario | 0.0544 |
| Mayor / Menor edad | 0.0265 : menos predictora |

Departamento es ahora la feature más importante (23.4%),
la geografía tiene mayor peso predictivo real.
Tipo de evento segunda (18.5%) — consistente con el EDA donde
Derrape y Atropello tienen las tasas de mortalidad más altas.

In [9]:
# -- Visualización 5: Distribución de probabilidades predichas ----------------------------------------------
df_proba = pd.DataFrame({
    'probabilidad': y_proba,
    'real': y_test.map({1:'Fallecido', 2:'Lesionado'})
})

fig_dist = px.histogram(
    df_proba,
    x='probabilidad',
    color='real',
    nbins=50,
    barmode='overlay',
    opacity=0.7,
    color_discrete_map={'Fallecido':'#E63946', 'Lesionado':'#5B8DEF'},
    title='Distribución de probabilidades predichas : Random Forest',
    labels={'probabilidad':'Probabilidad predicha (clase Fallecido)',
            'real':'Clase real'}
)
fig_dist.update_layout(height=420, template='plotly_white')
fig_dist.show()

### Análisis de distribución de probabilidades

Las distribuciones de Fallecido y Lesionado siguen solapándose
en la zona 0.3–0.7, lo cual es esperado dado que las features
son categóricas y el desbalance de clases es 4.2:1.
, la distribución es más honesta con los datos reales —
el modelo asigna probabilidades más conservadoras y distribuidas.

In [10]:
# -- Guardar resultados para comparación final ----------------------------------------------
resultados_rf = {
    'modelo'   : 'Random Forest',
    'accuracy' : round(acc, 4),
    'f1_score' : round(f1, 4),
    'roc_auc'  : round(auc, 4),
    'precision_fallecido': 0.32,
    'recall_fallecido'   : 0.65,
}

import json, os
os.makedirs('../data/models', exist_ok=True)
with open('../data/models/resultados_rf.json', 'w') as f:
    json.dump(resultados_rf, f, indent=2)

print('✓ Resultados guardados en data/models/resultados_rf.json')
print(f'\nResumen Random Forest:')
for k, v in resultados_rf.items():
    print(f'  {k:<25} {v}')

✓ Resultados guardados en data/models/resultados_rf.json

Resumen Random Forest:
  modelo                    Random Forest
  accuracy                  0.6949
  f1_score                  0.7233
  roc_auc                   0.7336
  precision_fallecido       0.32
  recall_fallecido          0.65


In [11]:
# -- Guardar modelo entrenado ----------------------------------------------
import joblib
import os

os.makedirs('../data/models', exist_ok=True)
joblib.dump(rf, '../data/models/random_forest.pkl')
print('✓ Modelo guardado en data/models/random_forest.pkl')

✓ Modelo guardado en data/models/random_forest.pkl


## 4. Resumen del modelo

| Métrica | Valor |
|---|---|
| Accuracy | 69.49% |
| F1-Score (weighted) | 72.33% |
| ROC-AUC | 73.36% |
| Precision Fallecido | 34% |
| Recall Fallecido | 62% |

**Conclusiones:**
- ROC-AUC mejoró de 0.7154 a 0.7336 al eliminar SMOTE
- Departamento es ahora la feature más predictora (23.4%)
- Importancia de features balanceada entre las 8 variables
- class_weight="balanced" maneja el desbalance sin contaminación